This notebook is made for data preprocessing. <div> Here i will present all of the steps i did to "clean" my data so its less of a headache when training</div>

In [27]:
import pandas as pd
import numpy as np
import os
from google.colab import drive
import shutil
import json
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
common_path = os.path.join("/content/drive/MyDrive","Train licenta","Train Dataset")
dataset_path = os.path.join(common_path,"batch_1_images")
label_path = os.path.join(common_path,"batch1_1.csv")
df_labels = pd.read_csv(label_path)
df_labels.sort_values(by='File Name',inplace=True)
df_labels.reset_index(drop=True,inplace=True)
print(len(df_labels))
print(len(os.listdir(dataset_path)))

1414
1489


As you can see, there is more unlabeled data. <p>I will move the unlabeled data into another folder and overwrite the original CSV </p>

In [12]:
all_images = pd.Series(os.listdir(dataset_path),name='File Name')
df_all_labels = df_labels.merge(all_images,
                                how='right',
                                on='File Name',
                                suffixes=("_original","_existing"))
missing_images = df_all_labels[df_all_labels['Json Data'].isna()]['File Name'].values

In [15]:
new_folder_path = os.path.join(common_path,"missing_images")
for idx,file_name in enumerate(os.listdir(dataset_path)):
  if file_name in missing_images:
    old_path = os.path.join(dataset_path,file_name)
    new_path = os.path.join(new_folder_path,file_name)
    shutil.move(old_path,new_folder_path)

Now we rename all images, so they are in sequential order

In [18]:
for idx,file_name in enumerate(sorted(os.listdir(dataset_path))):
  old_file = os.path.join(dataset_path,file_name)
  new_file = os.path.join(dataset_path,f'batch1-{idx+1:04d}.jpg')
  os.rename(old_file,new_file)

In [23]:
for idx,file_name in enumerate(df_labels['File Name'].values):
  df_labels.iloc[idx,0] = f'batch1-{idx+1:04d}.jpg'
df_labels['Json Data'] = df_labels['Json Data'].str.replace("\n","")

In [38]:
new_csv_path = os.path.join(common_path,"batch1_1_updated.csv")
df_labels.to_csv(new_csv_path)